# Assignment 2 — SimCLR on the 16-class scenes

Set the runtime to **GPU** before running anything (Runtime → Change runtime type → T4).
The full ladder is about two hours. Each training cell is separate so a disconnect does not restart finished runs.

This notebook expects the repository to already be on GitHub. If `git clone` fails, the local project has not been pushed yet.

In [ ]:
import torch
print('cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU. Change the runtime type to T4 and rerun this cell.')

In [ ]:
import os
if not os.path.exists('train.py'):
    if os.path.exists('/content/SSL_Assignment2/train.py'):
        os.chdir('/content/SSL_Assignment2')
    else:
        !git clone https://github.com/smritib4/SSL_Assignment2.git
        os.chdir('/content/SSL_Assignment2')
print(os.getcwd())
!pip install -q -r requirements.txt

## Data

Same Drive folder as Assignment 1. If you still have that zip, upload it and run `prepare_data.py --zip` instead of `gdown`.

In [ ]:
import os
if not os.path.isdir('data/train'):
    !pip install -q gdown
    !gdown --folder https://drive.google.com/drive/folders/1NWC3TMsXSWN2TeoYMCjhf2N1b-WRDh-M -O data_download --remaining-ok
    !python scripts/prepare_data.py --src data_download
else:
    print('data/train already exists')

## Ladder

Run the SmallCNN first. Assignment 1 scored that architecture at **89.38% validation**. If this run is far from that, stop. The later rows will not be interpretable.

Then the ResNet-18 baseline, SimCLR pretraining, the probe, the two fine-tunes, and joint training. Do not run the test cell until the table is printed.

In [ ]:
!python train.py --config configs/smallcnn_supervised.yaml

In [ ]:
!python train.py --config configs/resnet18_supervised.yaml

In [ ]:
!python train.py --config configs/simclr_pretrain.yaml

In [ ]:
!python train.py --config configs/simclr_linear_probe.yaml

In [ ]:
!python train.py --config configs/simclr_finetune_matched.yaml

In [ ]:
!python train.py --config configs/simclr_finetune.yaml

In [ ]:
!python train.py --config configs/joint.yaml

In [ ]:
!python scripts/make_report_table.py

## Test set, once

Put the leader printed above into the checkpoint path. If two finished runs are within one point, score both and report both. A gap that small was inside the noise floor in Assignment 1.

If the leader is not the joint run, edit `configs/best.yaml` so it matches the winner before you describe that file as the recipe.

In [ ]:
# Example. Replace the path with the leader from the table.
!python evaluate.py --checkpoint runs/joint_seed0/best.pt --split test

## Download the checkpoints and logs

Colab disk disappears with the session. Download this zip before you close the tab. Fill `reports/REPORT.md` from the table, and put a link to the best checkpoint in the README.

In [ ]:
!zip -r -q /content/ssl_runs.zip runs experiments/results experiments/run_logs
from google.colab import files
files.download('/content/ssl_runs.zip')